In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss
from functools import reduce
import datasets
import torch
from tqdm import tqdm
import os
from datetime import datetime

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


In [3]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_clicks_20230501"

dataset = load_from_disk(DATA_PATH)

In [5]:
polars_ds = dataset.to_polars()

In [19]:
index = faiss.read_index("data/complex_item_index_tiny.faiss")

with open("data/complex_item_ids", "rb") as fp:
    item_ids = pickle.load(fp)

In [7]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 569.92it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

train_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TRAIN_END_DT)
    .with_columns(
        pl.col("brand_name").fill_null(""),
        pl.col("item_condition_name").fill_null(""),
        pl.col("size_name").fill_null(""),
        pl.col("color").fill_null("")
    )
)

all_clicks = (
    train_interactions
    .select(
        pl.col("user_id"),
        pl.col("c2_name"),
        pl.col("name"),
        pl.col("item_id"),
        pl.col("stime"),
        pl.col("brand_name"),
        pl.col("item_condition_name"),
        pl.col("size_name"),
        pl.col("color"),
        pl.col("stime").rank("dense", descending=True).over("user_id").alias("rn")
    )
)

In [12]:
query_clicks_20 = (
    all_clicks
    .filter(pl.col("rn") <= 20)
    .select("item_id", "name", "brand_name", "item_condition_name", "size_name", "color")
    .unique()
)

In [17]:
item_names = query_clicks_20["name"].to_list()
ids = query_clicks_20["item_id"].to_list()
brand_names = query_clicks_20["brand_name"].to_list()
condition_names = query_clicks_20["item_condition_name"].to_list()
size_names = query_clicks_20["size_name"].to_list()
color_names = query_clicks_20["color"].to_list()

item_info = [
    (item_names[i] + " " + brand_names[i] + " " + condition_names[i] + " " + size_names[i] + " " + color_names[i]).strip() for
    i in range(len(item_names))
]

In [20]:
batch_size = 4096
similar_items = {}

replace_func = np.vectorize(lambda x: item_ids[x])

for i in tqdm(range(0, len(item_info), batch_size)):
    cur_names = item_info[i:i+batch_size]
    cur_ids = ids[i:i+batch_size]
    queries = model.encode(cur_names, batch_size=batch_size, normalize_embeddings=True)
    _, idx = index.search(queries, k=20)
    recs = replace_func(idx)
    cur_similar_items = {cur_ids[i]: recs[i, :][recs[i, :] != cur_ids[i]].tolist() for i in range(len(cur_ids))}
    similar_items = {**similar_items, **cur_similar_items}

100%|██████████| 392/392 [22:07<00:00,  3.39s/it]


In [21]:
with open("data/similar_items_lst20_top20_complex", "wb") as fp:
    pickle.dump(similar_items, fp)

In [6]:
#with open("data/similar_items_lst5_top50", "rb") as fp:
#    similar_items = pickle.load(fp)

In [22]:
def get_similar_items(row):
    return list(reduce(lambda x, y: x + y, [similar_items[item_id] for item_id in row["last_clicks"] if item_id in similar_items]))

In [23]:
user_last_clicks_20 = (
     all_clicks
    .filter(pl.col("rn") <= 20)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("last_clicks"))
)

In [24]:
recs_20 = (
    user_last_clicks_20
    .with_columns(
        pl.struct(["last_clicks"]).apply(get_similar_items).alias("recs")
    )
)

In [25]:
test_interactions = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") > TRAIN_END_DT)
    .filter(pl.col("date") <=  TEST_END_DT)
    .groupby("user_id")
    .agg(pl.col("item_id").alias("future_clicks"))
    .join(
        recs_20,
        on="user_id",
        how="inner"
    )
)

In [26]:
from replay.metrics import Recall, Precision, HitRate

In [27]:
TOP_K_VALUES = [10, 100, 400]

def calc_recall(row):
    return Recall._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_precision(row):
    return Precision._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def calc_hitrate(row):
    return HitRate._get_metric_value_by_user(TOP_K_VALUES, row["future_clicks"], row["recs"])

def intersection(row):
    return len(set(row["recs"]) & set(row["future_clicks"]))

metrics = (
    test_interactions
    .with_columns(
        pl.struct(["future_clicks", "recs"]).apply(calc_recall).alias("recall"),
        pl.struct(["future_clicks", "recs"]).apply(calc_precision).alias("precision"),
        pl.struct(["future_clicks", "recs"]).apply(calc_hitrate).alias("hitrate"),
    )
    .select(
        pl.col("recall").arr.get(0).mean().alias("recall@10"),
        pl.col("recall").arr.get(1).mean().alias("recall@100"),
        pl.col("recall").arr.get(2).mean().alias("recall@1000"),
        pl.col("precision").arr.get(0).mean().alias("precision@10"),
        pl.col("precision").arr.get(1).mean().alias("precision@100"),
        pl.col("precision").arr.get(2).mean().alias("precision@1000"),
        pl.col("hitrate").arr.get(0).mean().alias("hitrate@10"),
        pl.col("hitrate").arr.get(1).mean().alias("hitrate@100"),
        pl.col("hitrate").arr.get(2).mean().alias("hitrate@1000"),
        pl.col("hitrate").arr.get(0).sum().alias("hitrate_sum@10"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(1).sum().alias("hitrate_sum@100"), # количество рекомендаций, попавших в отложенную выборку
        pl.col("hitrate").arr.get(2).sum().alias("hitrate_sum@1000") # количество рекомендаций, попавших в отложенную выборку
    )
    .head(5)
)

In [28]:
metrics # поиск, используя весь контент

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007534,0.036734,0.072381,0.007458,0.004552,0.002868,0.055859,0.227338,0.389534,6427.0,26157.0,44819.0


In [41]:
metrics # поиск, используя только название

recall@10,recall@100,recall@1000,precision@10,precision@100,precision@1000,hitrate@10,hitrate@100,hitrate@1000,hitrate_sum@10,hitrate_sum@100,hitrate_sum@1000
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.007292,0.035024,0.067852,0.0071,0.004236,0.002637,0.052174,0.215291,0.371847,6003.0,24771.0,42784.0
